# Reproduce E2E — 저장된 실험 로드 후 예측만 수행

[experiments.json](../4_output/experiments/experiments.json) 에 저장된 파라미터를 읽어 **HPO 없이** 저장된 `best_params` 로 학습 1회만 돌려서 동일한 예측을 재현한다.

- 학습된 모델 객체가 저장돼 있지 않으므로 `rerun_best_trial` 로 한 번 재학습한다.
- 동일 SEED + 동일 파라미터이지만, 실행 환경(GPU/CPU, 라이브러리 버전)에 따라 미세한 차이가 날 수 있다.
- 현재는 `final_method='e2e_single'` 실험만 지원한다.

## 1. 환경 설정 및 데이터 로드

In [1]:
# ============================================================
# 환경 설정 + 데이터 로드 (e2e_twostage.ipynb와 동일)
# ============================================================
import os, sys

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/e2e_hpo.py'):
        os.system('gdown 1Vrn5LBl611rWbag7d09LZH68_lfpu6wP -O /content/modules.zip')
        os.makedirs('/content/project/3_modeling/modules', exist_ok=True)
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling/modules')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# --- 기본 라이브러리 ---
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- 프로젝트 유틸 ---
from utils.config import PROJECT_ROOT, OUTPUT_DIR, SEED, TARGET_COL, KEY_COL, POSITION_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import evaluate, rmse
from utils.experiment import download_from_drive

# --- 전처리 모듈 ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '2_preprocessing'))
from cleaning import run_cleaning
from outlier import run_outlier_treatment

# --- 3_modeling 모듈 ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '3_modeling'))
from modules.e2e_hpo import rerun_best_trial

# --- 데이터 로드 ---
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

setup 완료
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749
Feature 수: 1087
Die 수: train=104,988, val=34,996, test=34,996


## 2. 실험 설정 로드 (JSON → 파라미터 주입)

In [2]:
# ============================================================
# 실험 설정 로드 — JSON에서 파라미터 읽어와서 그대로 주입
# ============================================================

# ── 사용자 지정 ──
EXP_ID = '3-2-100'       # 재현할 실험번호
EVAL_TEST = False        # True: test RMSE 계산 (test peeking), False: val만

# ── Google Drive 파일 ID (Colab에서 JSON 최신본 다운로드용) ──
XLSX_GDRIVE_ID = '1IgaNh7ixgqpmH5PiwmSFbK2li6GODdew'
JSON_GDRIVE_ID = '1ycr6n5Ty_jzl4F-qQE4Cv5nS2WbIAZih'
download_from_drive(XLSX_GDRIVE_ID, JSON_GDRIVE_ID)

# ── JSON 로드 ──
json_path = os.path.join(OUTPUT_DIR, 'experiments', 'experiments.json')
with open(json_path, encoding='utf-8') as f:
    all_exps = json.load(f)

if EXP_ID not in all_exps:
    raise KeyError(
        f"실험번호 '{EXP_ID}' 가 {json_path} 에 없습니다. "
        f"사용 가능한 실험: {list(all_exps.keys())}"
    )

exp = all_exps[EXP_ID]

# ── 파라미터 추출 ──
cleaning_params = exp['cleaning_params']
outlier_params  = exp['outlier_params']
model_params    = exp['model_params']

pipeline_config = model_params['pipeline_config']
e2e_params      = model_params['e2e_params']
rerun_params    = model_params['rerun_params']
sampling_params = model_params['sampling_params']
LABEL_COL       = model_params.get('label_col', 'label_bin')
EXCLUDE_COLS    = model_params.get('exclude_cols', []) or []
best_params     = model_params['best_params']

# rerun_params 안의 튜플/리스트 정규화 (JSON은 튜플 → 리스트로 저장됨)
if isinstance(e2e_params.get('top_k_range'), list):
    e2e_params['top_k_range'] = tuple(e2e_params['top_k_range'])

# ── 현재 재현 노트북은 e2e_single만 지원 ──
final_method = model_params.get('final_method')
expected_val_rmse = model_params.get('final_val_rmse')
expected_test_rmse = model_params.get('final_test_rmse')

if final_method != 'e2e_single':
    raise NotImplementedError(
        f"실험 '{EXP_ID}'의 final_method={final_method!r} 는 현재 재현 노트북이 지원하지 않습니다.\n"
        f"이 노트북은 단일 E2E 모델(final_method='e2e_single')만 재현합니다.\n"
        f"앙상블 winner 재현이 필요하면 별도 확장이 필요합니다."
    )

# ── 설정 출력 ──
print(f'재현 대상       : {EXP_ID}')
print(f'final_method    : {final_method}')
print(f'CLF / REG       : {e2e_params["clf_model"]} / {e2e_params["reg_model"]}')
print(f'Rerun mode      : {rerun_params["mode"]} (n_folds={rerun_params.get("n_folds")})')
print(f'Top-K (fixed)   : {e2e_params["top_k_fixed"]}')
print(f'샘플링          : {"ON" if sampling_params["use_sampling"] else "OFF"}')
print(f'EVAL_TEST       : {EVAL_TEST}')
print(f'─── 저장된 결과 ───')
print(f'final_val_rmse  : {expected_val_rmse}')
print(f'final_test_rmse : {expected_test_rmse}')
print(f'\n── Pipeline Config ──')
for k, v in pipeline_config.items():
    print(f'  {k}: {v}')

재현 대상       : 3-2-100
final_method    : e2e_single
CLF / REG       : lgbm / lgbm
Rerun mode      : kfold (n_folds=5)
Top-K (fixed)   : 200
샘플링          : OFF
EVAL_TEST       : False
─── 저장된 결과 ───
final_val_rmse  : 0.005810358584387139
final_test_rmse : None

── Pipeline Config ──
  input_level: die
  run_clf: True
  clf_output: proba
  clf_filter: False
  clf_optuna: True
  run_fs: False
  fs_optuna: False
  reg_level: position
  reg_optuna: True


## 3. 전처리 (저장된 cleaning / outlier 파라미터로 재실행)

In [3]:
# --- Step 1: 클리닝 ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    **cleaning_params,
)

print(f'\n클리닝 결과: {len(feat_cols)} -> {len(clean_cols)}개')

# --- Step 2: 이상치 처리 ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    **outlier_params,
)

# --- Step 3: 웨이퍼맵 기반 피처 필터 ---
if EXCLUDE_COLS:
    before = len(clean_cols)
    clean_cols = [c for c in clean_cols if c not in set(EXCLUDE_COLS)]
    print(f'\n웨이퍼맵 필터: {before} -> {len(clean_cols)}개 ({before - len(clean_cols)}개 제외)')

print(f'\n최종 피처 수: {len(clean_cols)}개')

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=25%
  제거: 10개, 잔여: 972개
    컬럼: 982 → 972 (10개 제거)
    DataFrame: (104988, 976)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 946개
    컬럼: 972 → 946 (26개 제거)
    DataFrame: (104988, 950)



NameError: name 'y_train' is not defined

## 4. Health Merge + 라벨 생성 + Position 분리

In [ ]:
# --- Health merge ---
die_train = xs_train.merge(ys['train'], on=KEY_COL, how='left')
die_val = xs_val.merge(ys['validation'], on=KEY_COL, how='left')
die_test = xs_test.merge(ys['test'], on=KEY_COL, how='left')

assert die_train[TARGET_COL].notna().all(), 'train health NaN!'
assert die_val[TARGET_COL].notna().all(), 'val health NaN!'
assert die_test[TARGET_COL].notna().all(), 'test health NaN!'
print(f'Die-level merge: train={die_train.shape}, val={die_val.shape}, test={die_test.shape}')

# --- 샘플링 (unit 기준, train만) — JSON의 sampling_params 그대로 사용 ---
USE_SAMPLING = sampling_params['use_sampling']
SAMPLE_FRAC = sampling_params['sample_frac']

if USE_SAMPLING:
    all_units = die_train[KEY_COL].drop_duplicates()
    sampled_units = all_units.sample(frac=SAMPLE_FRAC, random_state=SEED)
    n_before = len(die_train)
    die_train = die_train[die_train[KEY_COL].isin(sampled_units)].reset_index(drop=True)
    print(f'\n샘플링 ON (frac={SAMPLE_FRAC})')
    print(f'  Unit: {len(all_units):,} -> {len(sampled_units):,}')
    print(f'  Die: {n_before:,} -> {len(die_train):,}')
else:
    print('\n샘플링 OFF -> 전체 데이터 사용')

# --- 라벨 생성: 0 vs >0 ---
for df in [die_train, die_val, die_test]:
    df[LABEL_COL] = (df[TARGET_COL] > 0).astype(int)

print(f'\n라벨 분포 (train):')
dist = die_train[LABEL_COL].value_counts().sort_index()
for k, v in dist.items():
    print(f'  {k}: {v:,} ({v / len(die_train) * 100:.1f}%)')

# --- Position별 분리 ---
positions = sorted(die_train[POSITION_COL].unique())
feat_cols_clean = clean_cols

pos_data = {}
for pos in positions:
    pos_data[pos] = {
        'train': die_train[die_train[POSITION_COL] == pos].reset_index(drop=True),
        'val':   die_val[die_val[POSITION_COL] == pos].reset_index(drop=True),
        'test':  die_test[die_test[POSITION_COL] == pos].reset_index(drop=True),
    }
    n = {k: len(v) for k, v in pos_data[pos].items()}
    print(f'Position {pos}: train={n["train"]:,}, val={n["val"]:,}, test={n["test"]:,}')

print(f'\nClean feature 수: {len(feat_cols_clean)}개')

## 5. 저장된 best_params 로 학습 1회 (HPO 스킵)

In [ ]:
# ============================================================
# HPO 스킵 → 저장된 best_params로 학습 1회 (rerun_best_trial)
# ============================================================
# _build_params_from_best 가 clf_* / reg_* prefix 를 내부에서 분리하므로
# JSON의 best_params 를 그대로 넘기면 된다.

final = rerun_best_trial(
    pos_data=pos_data,
    feat_cols=feat_cols_clean,
    best_params=best_params,
    pipeline_config=pipeline_config,
    clf_model=e2e_params['clf_model'],
    reg_model=e2e_params['reg_model'],
    label_col=LABEL_COL,
    imbalance_method=e2e_params['imbalance_method'],
    agg_funcs=e2e_params['agg_funcs'],
    top_k_fixed=e2e_params['top_k_fixed'],
    clf_fixed=e2e_params['clf_fixed'],
    reg_fixed=e2e_params['reg_fixed'],
    **rerun_params,
)

# ── Val 평가 ──
# ⚠️ JSON 의 final_val_rmse 는 rerun_best_trial 내부의 rmse() (UNCLIPPED) 값.
#    evaluate() 는 기본 clip=True 라 음수 예측이 있으면 다른 값을 출력한다.
#    JSON 비교는 반드시 UNCLIPPED(= final['val_rmse']) 로 해야 한다.
y_val = final['unit_data']['val'][TARGET_COL].values
repro_val_rmse = float(final['val_rmse'])   # unclipped — JSON 과 apples-to-apples
print(f'[Reproduce {EXP_ID} (val)] UNCLIPPED RMSE = {repro_val_rmse:.8f}  '
      f'(n={len(y_val):,}, zero={(y_val == 0).sum():,})')
# 참고용: clipped RMSE (제출 CSV 후처리 시의 값)
evaluate(y_val, final['val_pred'], label=f'Reproduce {EXP_ID} (val, CLIPPED 참고)')

# ── Test 평가 (EVAL_TEST=True 일 때만) ──
if EVAL_TEST:
    y_test = final['unit_data']['test'][TARGET_COL].values
    repro_test_rmse = float(rmse(y_test, final['test_pred']))   # unclipped
    print(f'[Reproduce {EXP_ID} (test)] UNCLIPPED RMSE = {repro_test_rmse:.8f}  '
          f'(n={len(y_test):,}, zero={(y_test == 0).sum():,})')
    evaluate(y_test, final['test_pred'], label=f'Reproduce {EXP_ID} (test, CLIPPED 참고)')
else:
    repro_test_rmse = None
    print('[test] EVAL_TEST=False → test RMSE 계산 생략 (test peeking 방지)')

## 6. 재현 검증 — 저장된 RMSE vs 재현 RMSE

In [ ]:
# ============================================================
# 재현 검증 — 저장된 RMSE 와 재현 RMSE 비교
# ============================================================
# 같은 파라미터 + 같은 SEED 이지만 환경(GPU/CPU/라이브러리 버전) 차이로
# bit-identical 이 아닐 수 있다. 아래 3단계로 판단한다.

TOL_EXACT = 1e-8
TOL_CLOSE = 1e-4

def _report(name, expected, reproduced):
    print(f'── {name} ──')
    if expected is None:
        print(f'  기대값: N/A (JSON에 기록 없음)')
        print(f'  재현값: {reproduced:.8f}' if reproduced is not None else '  재현값: N/A')
        return
    if reproduced is None:
        print(f'  기대값: {expected:.8f}')
        print(f'  재현값: N/A (EVAL_TEST=False)')
        return
    diff = reproduced - expected
    rel = abs(diff) / max(abs(expected), 1e-12)
    print(f'  기대값: {expected:.8f}')
    print(f'  재현값: {reproduced:.8f}')
    print(f'  차이  : {diff:+.2e}  (상대 {rel:.2e})')
    if abs(diff) < TOL_EXACT:
        print(f'  ✓ 완전 일치 (< {TOL_EXACT:.0e})')
    elif abs(diff) < TOL_CLOSE:
        print(f'  ○ 근사 일치 (< {TOL_CLOSE:.0e}) — 결정성 차이 범위')
    else:
        print(f'  ⚠ 불일치 — 환경/버전/GPU-CPU 차이 의심')

_report('Val RMSE', expected_val_rmse, repro_val_rmse)
if EVAL_TEST:
    _report('Test RMSE', expected_test_rmse, repro_test_rmse)

## 7. 예측 결과 CSV 저장

In [ ]:
# ============================================================
# 예측 결과 CSV 저장 (기존 파일 보호: _reproduce 접미사)
# ============================================================
key_val  = final['unit_data']['val'][KEY_COL].values
key_test = final['unit_data']['test'][KEY_COL].values

repro_val_path  = os.path.join(OUTPUT_DIR, f'{EXP_ID}_reproduce_val.csv')
repro_test_path = os.path.join(OUTPUT_DIR, f'{EXP_ID}_reproduce_test.csv')

pd.DataFrame({KEY_COL: key_val,  TARGET_COL: final['val_pred'] }).to_csv(repro_val_path,  index=False)
pd.DataFrame({KEY_COL: key_test, TARGET_COL: final['test_pred']}).to_csv(repro_test_path, index=False)

print(f'[재현 예측 저장]')
print(f'  val  → {repro_val_path}')
print(f'  test → {repro_test_path}')
print()
print(f'완료: 실험 {EXP_ID} 재현')
print(f'  재현 val_rmse: {repro_val_rmse:.8f}')
if expected_val_rmse is not None:
    print(f'  저장 val_rmse: {expected_val_rmse:.8f}')